In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras


In [2]:
# Function to load the data from file
def load_data(filename):
    sentences = []
    labels = []
    with open(filename, 'r', encoding='utf-8') as file:
        sentence = []
        label = []
        for line in file:
            if line == '\n':
                if sentence:
                    sentences.append(sentence)
                    labels.append(label)
                    sentence = []
                    label = []
                continue
            token, entity, _ = line.strip().split('\t')
            sentence.append(token)
            label.append(entity)
    return sentences, labels

In [3]:
# Load the training and test data
train_sentences, train_labels = load_data('training.conll')
test_sentences, test_labels = load_data('test.conll')

In [5]:
# Tokenization and Preprocessing
tokenizer = keras.preprocessing.text.Tokenizer(lower=True, oov_token='OOV')
tokenizer.fit_on_texts(train_sentences)
vocab_size = len(tokenizer.word_index) + 1

train_sequences = tokenizer.texts_to_sequences(train_sentences)
test_sequences = tokenizer.texts_to_sequences(test_sentences)

max_len = max(len(seq) for seq in train_sequences + test_sequences)

X_train = keras.preprocessing.sequence.pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_test = keras.preprocessing.sequence.pad_sequences(test_sequences, maxlen=max_len, padding='post')

In [6]:
# Convert labels to one-hot encoding
label_encoder = keras.preprocessing.text.Tokenizer(lower=False)
label_encoder.fit_on_texts(train_labels)

num_labels = len(label_encoder.word_index) + 1

y_train = keras.preprocessing.sequence.pad_sequences(label_encoder.texts_to_sequences(train_labels), maxlen=max_len, padding='post')
y_train = keras.utils.to_categorical(y_train, num_labels)

y_test = keras.preprocessing.sequence.pad_sequences(label_encoder.texts_to_sequences(test_labels), maxlen=max_len, padding='post')
y_test = keras.utils.to_categorical(y_test, num_labels)

In [7]:
# Split training data into training and validation sets
validation_split = 0.2
split_index = int(len(X_train) * (1 - validation_split))

X_train, X_val = X_train[:split_index], X_train[split_index:]
y_train, y_val = y_train[:split_index], y_train[split_index:]

In [8]:
# Step 2: Word Embedding
embedding_dim = 100

model = keras.Sequential([
    keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_len),
    keras.layers.Bidirectional(keras.layers.LSTM(units=64, return_sequences=True)),
    keras.layers.Dense(num_labels, activation='softmax')
])


In [9]:
# Step 3: Model Compilation
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [10]:
# Step 4: Model Training
epochs = 10
batch_size = 32

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size)


Epoch 1/10
69/69 [==============================] - 26s 195ms/step - loss: 0.4724 - accuracy: 0.9109 - val_loss: 0.0873 - val_accuracy: 0.9851
Epoch 2/10
69/69 [==============================] - 10s 141ms/step - loss: 0.0640 - accuracy: 0.9897 - val_loss: 0.0565 - val_accuracy: 0.9927
Epoch 3/10
69/69 [==============================] - 11s 154ms/step - loss: 0.0497 - accuracy: 0.9926 - val_loss: 0.0521 - val_accuracy: 0.9921
Epoch 4/10
69/69 [==============================] - 11s 154ms/step - loss: 0.0424 - accuracy: 0.9934 - val_loss: 0.0471 - val_accuracy: 0.9920
Epoch 5/10
69/69 [==============================] - 10s 151ms/step - loss: 0.0357 - accuracy: 0.9935 - val_loss: 0.0416 - val_accuracy: 0.9931
Epoch 6/10
69/69 [==============================] - 11s 158ms/step - loss: 0.0282 - accuracy: 0.9936 - val_loss: 0.0378 - val_accuracy: 0.9936
Epoch 7/10
69/69 [==============================] - 11s 153ms/step - loss: 0.0221 - accuracy: 0.9936 - val_loss: 0.0362 - val_accuracy: 0.9933

In [11]:
# Step 5: Model Evaluation
_, accuracy = model.evaluate(X_test, y_test, batch_size=batch_size)
print('Test Accuracy:', accuracy)


14/14 [==============================] - 1s 49ms/step - loss: 0.0583 - accuracy: 0.9890
Test Accuracy: 0.9890410900115967


In [29]:
import numpy as np
from sklearn.metrics import f1_score

# Perform NER prediction
predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=-1)

# Flatten the predicted_labels array
predicted_labels_flat = predicted_labels.flatten()

# Reshape X_test to match the dimensions of predicted_labels_flat
X_test_reshaped = np.reshape(X_test, (X_test.shape[0] * X_test.shape[1],))

# Ensure the lengths of X_test_reshaped and predicted_labels_flat are the same
min_len = min(len(X_test_reshaped), len(predicted_labels_flat))
X_test_reshaped = X_test_reshaped[:min_len]
predicted_labels_flat = predicted_labels_flat[:min_len]

# Convert y_test to one-hot encoding
y_true = np.argmax(y_test[:min_len], axis=-1).flatten()

# Compute the F1 score for the NER task with minimum span 0 and maximum span 4
f1_score_span_0_4 = f1_score(y_true, predicted_labels_flat, average='micro')
print('F1 Score (Span 0-4):', f1_score_span_0_4)


14/14 [==============================] - 1s 47ms/step
F1 Score (Span 0-4): 0.989041095890411
